# 10. Evaluate Localization-only GT-E2E

기존 semantic `baseline`/`shared_e2e`와 localization-only 모델을 동일한 class-agnostic 조건으로 비교합니다. 모든 prediction/GT label을 `object=0`으로 통일하므로 semantic class 정답 여부가 localization 지표에 섞이지 않습니다.

두 종류의 결과를 함께 봅니다.

- `loc_mAP/AP50/AP75/AR100`: score ranking까지 포함한 실제 localization 성능
- `matched IoU`, center/size error, geometry recall: top-100 query box 자체의 기하 품질

In [1]:
from pathlib import Path
import gc, importlib, sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (Path('D:/gt-super') / 'data').exists():
    ROOT = Path('D:/gt-super')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import gt_aux.config as config_module
import gt_aux.data as data_module
import gt_aux.model as model_module
import gt_aux.eval as eval_module
config_module = importlib.reload(config_module)
data_module = importlib.reload(data_module)
model_module = importlib.reload(model_module)
eval_module = importlib.reload(eval_module)

ExperimentConfig = config_module.ExperimentConfig
prepare_data, make_loaders = data_module.prepare_data, data_module.make_loaders
load_checkpoint = eval_module.load_checkpoint
evaluate_main = eval_module.evaluate_main
collect_localization_predictions = eval_module.collect_localization_predictions
localization_metrics = eval_module.localization_metrics
localization_geometry_metrics = eval_module.localization_geometry_metrics

d:\gt-super\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RUN_MODE = 'smoke'
SEEDS = [42, 43]
DATA_SEED = 42
EXPERIMENTS = [
    'baseline',
    'shared_e2e',
    'localization_only',
    'localization_gt_e2e_aux010',
    'localization_gt_e2e_aux025',
    'localization_gt_e2e_aux050',
]
SEMANTIC_EXPERIMENTS = {'baseline', 'shared_e2e'}
TRAIN_IMAGES, VAL_IMAGES, EPOCHS = 400, 100, 7
BATCH_SIZE, IMAGE_MIN_SIZE, IMAGE_MAX_SIZE = 2, 384, 640
STRICT_CHECKPOINTS = True

CONFIG = ExperimentConfig.for_run(
    ROOT, run_mode=RUN_MODE, seed=SEEDS[0], data_seed=DATA_SEED,
    experiments=EXPERIMENTS, train_images=TRAIN_IMAGES, val_images=VAL_IMAGES,
    epochs=EPOCHS, batch_size=BATCH_SIZE, image_min_size=IMAGE_MIN_SIZE,
    image_max_size=IMAGE_MAX_SIZE, feature_level=0,
)
CONFIG.as_dict()

{'root': 'D:\\gt-super',
 'run_mode': 'smoke',
 'checkpoint': 'SenseTime/deformable-detr',
 'train_images': 400,
 'val_images': 100,
 'epochs': 7,
 'batch_size': 2,
 'num_workers': 0,
 'image_size': {'shortest_edge': 384, 'longest_edge': 640},
 'lr': 0.0002,
 'backbone_lr': 2e-05,
 'weight_decay': 0.0001,
 'grad_clip': 0.1,
 'base_aux_weight': 0.5,
 'feature_level': 0,
 'horizontal_flip_p': 0.5,
 'use_amp': True,
 'deterministic': True,
 'save_epoch_checkpoints': False,
 'device': 'cuda',
 'experiments': ['baseline',
  'shared_e2e',
  'localization_only',
  'localization_gt_e2e_aux010',
  'localization_gt_e2e_aux025',
  'localization_gt_e2e_aux050'],
 'seed': 42,
 'data_seed': 42}

In [3]:
BUNDLE = prepare_data(CONFIG)
checkpoint_rows = []
for experiment in EXPERIMENTS:
    for seed in SEEDS:
        path = CONFIG.checkpoint_path(experiment, seed)
        checkpoint_rows.append({
            'experiment': experiment, 'seed': seed, 'exists': path.exists(), 'path': str(path)
        })
checkpoint_df = pd.DataFrame(checkpoint_rows)
display(checkpoint_df)
missing = checkpoint_df.loc[~checkpoint_df['exists']]
if STRICT_CHECKPOINTS and not missing.empty:
    raise FileNotFoundError(
        'Missing checkpoints. Run 01_train_experiments.ipynb and ' +
        '01-3_train_localization_only_gt_e2e.ipynb first:\n' +
        '\n'.join(missing['path'])
    )

VOC XML: 100%|██████████| 3750/3750 [00:01<00:00, 3019.73it/s]

Full split: train=3000 (9180 objects), val=750 (2530 objects)
Current run: train=400, val=100


,experiment,seed,exists,path
0,baseline,42,True,D:\gt-super\cache\checkpoints\checkpoint_smoke...
1,baseline,43,True,D:\gt-super\cache\checkpoints\checkpoint_smoke...
2,shared_e2e,42,True,D:\gt-super\cache\checkpoints\checkpoint_smoke...
3,shared_e2e,43,True,D:\gt-super\cache\checkpoints\checkpoint_smoke...
4,localization_only,42,True,D:\gt-super\cache\checkpoints\checkpoint_smoke...
5,localization_only,43,True,D:\gt-super\cache\checkpoints\checkpoint_smoke...
6,localization_gt_e2e_aux010,42,False,D:\gt-super\cache\checkpoints\checkpoint_smoke...
7,localization_gt_e2e_aux010,43,False,D:\gt-super\cache\checkpoints\checkpoint_smoke...
8,localization_gt_e2e_aux025,42,False,D:\gt-super\cache\checkpoints\checkpoint_smoke...
9,localization_gt_e2e_aux025,43,False,D:\gt-super\cache\checkpoints\checkpoint_smoke...


FileNotFoundError: Missing checkpoints. Run 01_train_experiments.ipynb and 01-3_train_localization_only_gt_e2e.ipynb first:
D:\gt-super\cache\checkpoints\checkpoint_smoke_localization_gt_e2e_aux010_seed42.pt
D:\gt-super\cache\checkpoints\checkpoint_smoke_localization_gt_e2e_aux010_seed43.pt
D:\gt-super\cache\checkpoints\checkpoint_smoke_localization_gt_e2e_aux025_seed42.pt
D:\gt-super\cache\checkpoints\checkpoint_smoke_localization_gt_e2e_aux025_seed43.pt
D:\gt-super\cache\checkpoints\checkpoint_smoke_localization_gt_e2e_aux050_seed42.pt
D:\gt-super\cache\checkpoints\checkpoint_smoke_localization_gt_e2e_aux050_seed43.pt

In [ ]:
rows, semantic_rows = [], []
available = checkpoint_df.loc[checkpoint_df['exists'], ['experiment', 'seed']]
for experiment, seed in available.itertuples(index=False):
    print(f'[{experiment} / seed={seed}]')
    _, val_loader = make_loaders(CONFIG, BUNDLE, seed)
    model, checkpoint, fingerprint = load_checkpoint(CONFIG, BUNDLE, experiment, seed)
    predictions = targets = None
    try:
        predictions, targets = collect_localization_predictions(
            model, val_loader, BUNDLE.processor, CONFIG
        )
        row = {
            'experiment': experiment, 'seed': seed,
            'checkpoint_epoch': checkpoint.get('epoch'), 'fingerprint': fingerprint,
            **localization_metrics(predictions, targets),
            **localization_geometry_metrics(predictions, targets, max_detections=100),
        }
        rows.append(row)
        if experiment in SEMANTIC_EXPERIMENTS:
            semantic_rows.append({
                'experiment': experiment, 'seed': seed,
                **evaluate_main(model, val_loader, BUNDLE.processor, CONFIG),
            })
        print({key: round(value, 4) for key, value in row.items()
               if key in {'loc_map', 'loc_ap50', 'loc_ap75', 'loc_ar100',
                          'matched_iou', 'center_error_l2', 'size_error_l1'}})
    finally:
        model.close()
        model.cpu()
        del model, predictions, targets
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

results_df = pd.DataFrame(rows)
semantic_df = pd.DataFrame(semantic_rows)
raw_path = CONFIG.output_dir / 'localization_only_comparison_raw.csv'
semantic_path = CONFIG.output_dir / 'localization_only_semantic_reference.csv'
results_df.to_csv(raw_path, index=False)
semantic_df.to_csv(semantic_path, index=False)
print('Saved:', raw_path)
print('Saved:', semantic_path)

In [ ]:
METRICS = [
    'loc_map', 'loc_ap50', 'loc_ap75', 'loc_ar100',
    'loc_ap_small', 'loc_ap_medium', 'loc_ap_large',
    'matched_iou', 'center_error_l2', 'size_error_l1',
    'geometry_recall50', 'geometry_recall75',
]
summary = results_df.groupby('experiment')[METRICS].agg(['mean', 'std', 'count'])
summary_path = CONFIG.output_dir / 'localization_only_comparison_summary.csv'
summary.to_csv(summary_path)
display(summary)
print('Saved:', summary_path)

# 같은 seed끼리 뺀 paired delta: 양수 우위 지표와 음수 우위 error 지표를 그대로 보존합니다.
delta_frames = []
for reference in ['baseline', 'shared_e2e', 'localization_only']:
    reference_df = results_df.query('experiment == @reference').set_index('seed')[METRICS]
    for experiment in results_df['experiment'].unique():
        if experiment == reference:
            continue
        candidate = results_df.query('experiment == @experiment').set_index('seed')[METRICS]
        common = candidate.index.intersection(reference_df.index)
        if common.empty:
            continue
        delta = candidate.loc[common] - reference_df.loc[common]
        delta['experiment'] = experiment
        delta['reference'] = reference
        delta['seed'] = common
        delta_frames.append(delta.reset_index(drop=True))
paired_delta_df = pd.concat(delta_frames, ignore_index=True)
delta_path = CONFIG.output_dir / 'localization_only_paired_deltas.csv'
paired_delta_df.to_csv(delta_path, index=False)
display(paired_delta_df.groupby(['reference', 'experiment'])[METRICS].mean())
print('Saved:', delta_path)

In [ ]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(2, 3, figsize=(22, 12))
plot_specs = [
    ('loc_map', 'Class-agnostic mAP', False),
    ('loc_ap75', 'Class-agnostic AP75', False),
    ('loc_ar100', 'Class-agnostic AR100', False),
    ('matched_iou', 'Matched box IoU', False),
    ('center_error_l2', 'Normalized center error (lower is better)', True),
    ('size_error_l1', 'Normalized size L1 (lower is better)', True),
]
for axis, (metric, title, lower_is_better) in zip(axes.flat, plot_specs):
    sns.barplot(data=results_df, x='experiment', y=metric, errorbar='sd', ax=axis)
    sns.stripplot(data=results_df, x='experiment', y=metric, color='black', size=4, ax=axis)
    axis.set_title(title)
    axis.tick_params(axis='x', rotation=28)
plt.tight_layout()
figure_path = CONFIG.output_dir / 'localization_only_comparison.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
print('Saved:', figure_path)
plt.show()

In [ ]:
scale_df = results_df.melt(
    id_vars=['experiment', 'seed'],
    value_vars=['loc_ap_small', 'loc_ap_medium', 'loc_ap_large'],
    var_name='scale', value_name='AP',
)
scale_df['scale'] = scale_df['scale'].str.replace('loc_ap_', '', regex=False)
fig, axis = plt.subplots(figsize=(13, 6))
sns.barplot(data=scale_df, x='scale', y='AP', hue='experiment', errorbar='sd', ax=axis)
axis.set_title('Class-agnostic AP by object scale')
plt.tight_layout()
scale_path = CONFIG.output_dir / 'localization_only_scale_ap.png'
fig.savefig(scale_path, dpi=180, bbox_inches='tight')
print('Saved:', scale_path)
plt.show()

print('Semantic detection reference (localization-only 모델에는 정의되지 않음)')
display(semantic_df.groupby('experiment')[['map', 'map50', 'map75', 'mar100']].agg(['mean', 'std']))

## 판정 기준

제안 효과는 먼저 `localization_gt_e2e_* - localization_only`의 paired delta로 판단합니다. AP/AR/matched IoU/recall은 양수, center·size error는 음수이면 개선입니다. 그다음 `baseline`과 `shared_e2e` 대비 절대 수준을 확인합니다. `lambda_aux` 선택은 평균 성능뿐 아니라 seed 표준편차와 학습 노트북의 aux/main gradient norm ratio(권장 0.3~0.7)를 함께 봅니다.